In [ ]:
import torch
import torch.nn as nn
import torch.nn.functional as F
import torch.optim as optim
from tqdm import trange
from trainui.client import Tracker, SequenceError   

torch.set_float32_matmul_precision('high')  # or 'medium'

DEVICE = "cuda"
DATA = open("shakespeare.txt").read()

VOCAB = list(sorted(set(DATA)))
VOCAB_MAP = {v:k for k,v in enumerate(VOCAB)}
VOCAB_SIZE = len(VOCAB)

def encode(data):
    return torch.tensor([VOCAB_MAP[x] for x in data])

def decode(t):
    return ''.join(VOCAB[x] for x in t)

DATA = encode(DATA).to(DEVICE)
DATA_TRAIN = DATA[:int(0.9 * DATA.size(0))]
DATA_TEST = DATA[int(0.9 * DATA.size(0)):]

@torch.no_grad()
def get_batch(data, width, batch_size):
    idx = torch.randint(data.size(0)-width,(batch_size,1), device=data.device)
    idx = idx + torch.arange(width+1, device=data.device)
    return data[idx[:,:-1]],data[idx[:,1:]]

class MultiheadAttention(nn.Module):
    def __init__(self, width, embeddings, headsize, dropout_p=0.0):
        super().__init__()
        assert embeddings % headsize == 0, "Embeddings should be multiple of headsize"
        self.headsize = headsize
        self.headcount = embeddings // headsize
        self.dropout_p = dropout_p
        self.qkv = nn.Linear(embeddings, 3*embeddings, bias=False)
        #self.drop = nn.Dropout(dropout_p)
        self.proj = nn.Linear(embeddings, embeddings)
        #self.register_buffer("causal_mask", torch.tril(torch.ones((width, width), dtype=bool)))
        ang = torch.arange(0, width).view(-1,1) * torch.tensor([1000.0**(-2*i/headsize) for i in range(headsize//2)])
        self.register_buffer("rope_cos", torch.cos(ang))
        self.register_buffer("rope_sin", torch.sin(ang))

    def rope(self, x):
        N = x.shape[-2]
        xx, yy = x.chunk(2,-1)
        return torch.concat(
            [xx*self.rope_cos[:N,:] - yy*self.rope_sin[:N,:],
            xx*self.rope_sin[:N,:] + yy*self.rope_cos[:N,:]],
            dim=-1)

    def forward(self, x):
        B,T,E = x.shape
        H,W = self.headcount, self.headsize
        q,k,v = self.qkv(x).view(B,T,3,H,W).permute(2,0,3,1,4) # B,H,T,W
        q = self.rope(q)
        k = self.rope(k)
        #a = (q @ k.transpose(-2,-1)) * W**-0.5 # B,H,T,T
        #a = torch.masked_fill(a, ~self.causal_mask[:T,:T], float("-inf"))
        #a = F.softmax(a, dim=-1)
        #a = self.drop(a)
        #a = (a @ v)
        a = F.scaled_dot_product_attention(q,k,v, is_causal=True, dropout_p=(self.dropout_p if self.training else 0.0))
        a = a.transpose(-3,-2).reshape(B,T,E)
        return self.proj(a)

class FeedForward(nn.Module):
    def __init__(self, embeddings, dropout_p=0.0):
        super().__init__()
        self.lin1 = nn.Linear(embeddings, embeddings*4)
        self.gelu = nn.GELU()
        self.lin2 = nn.Linear(embeddings*4, embeddings)

    def forward(self, x):
        x = self.lin1(x)
        x = self.gelu(x)
        x = self.lin2(x)
        return x

class AttentionBlock(nn.Module):
    def __init__(self, width, embeddings, headsize, dropout_p=0.0):
        super().__init__()
        self.mha = MultiheadAttention(
            width=width, 
            embeddings=embeddings, 
            headsize=headsize, 
            dropout_p=dropout_p)
        self.ff = FeedForward(
            embeddings=embeddings,
            dropout_p=dropout_p)
        self.layn1 = nn.LayerNorm(embeddings)
        self.layn2 = nn.LayerNorm(embeddings)
        self.drop = nn.Dropout(dropout_p)

    def forward(self, x):
        x = x + self.drop(self.mha(self.layn1(x)))
        x = x + self.drop(self.ff(self.layn2(x)))
        return x

class Transformer(nn.Module):
    def __init__(self, vocab_size, width, embeddings, blocks, headsize, dropout_p=0.0):
        super().__init__()
        self.vocab_size = vocab_size
        self.context_width = width
        self.embed = nn.Embedding(vocab_size, embeddings)
        #self.embed_pos = nn.Embedding(width, embeddings)
        self.drop = nn.Dropout(dropout_p)
        self.blocks = nn.Sequential(*[
            AttentionBlock(
                    width=width,
                    embeddings=embeddings,
                    headsize=headsize,
                    dropout_p=dropout_p) 
                for _ in range(blocks)
        ])
        self.layn = nn.LayerNorm(embeddings)
        self.lmh = nn.Linear(embeddings, vocab_size, bias=False)
        self.embed.weight = self.lmh.weight
        #self.register_buffer("position_offsets", torch.arange(width))

    def forward(self, x, y=None):
        B,T = x.shape
        x = self.embed(x)# + self.embed_pos(self.position_offsets[:T])
        x = self.drop(x)
        x = self.blocks(x)
        x = self.layn(x)
        logits = self.lmh(x)
        if y is None:
            return logits
        else:
            return F.cross_entropy(logits.view(-1,self.vocab_size), y.view(-1))

    @torch.no_grad()
    def generate(self, limit, prompt=""):
        was_training = self.training
        self.eval()
        out = []
        if prompt: 
            out.extend(encode(prompt).aslist())
        else:
            out.append(0)
        for _ in range(limit):
            logits = self(torch.tensor(out[-self.context_width:], device = self.lmh.weight.device).view(1,-1))
            logits = logits[0,-1,:]
            prob = F.softmax(logits, dim=-1)
            out.append(torch.multinomial(prob, num_samples=1))
        self.train(was_training)
        return out if prompt else out[1:]


@torch.no_grad()
def estimate_loss(model, batch_size=128):
    was_train = model.training
    model.eval()
    l_test, l_train = [
        model(*get_batch(data, model.context_width, batch_size)) for data in [DATA_TEST, DATA_TRAIN] 
    ]
    model.train(was_train)
    return  l_test, l_train


m = Transformer(
    vocab_size=VOCAB_SIZE,
    width=64,
    embeddings=128,
    blocks=8,
    headsize=32,
    dropout_p=0.1
).to(DEVICE)
m = torch.compile(m)

def train(model, optimizer, run, iterations, batch_size):
    pbar = trange(0,iterations, desc="Learning")
    for i in pbar:
        optimizer.zero_grad()
        with torch.autocast('cuda', dtype=torch.bfloat16):
            loss = model(*get_batch(DATA_TRAIN, model.context_width, batch_size))
        loss.backward()
        grad_norm = torch.nn.utils.clip_grad_norm_(model.parameters(), 1.0)
        optimizer.step()
        run.log(iteration=i, batches=batch_size, train_loss=loss.item(), lr =optimizer.param_groups[0]['lr'], grad_norm=grad_norm.item()) 
        if i%100==99:
            l_ts,l_tr = estimate_loss(model)
            pbar.set_postfix({
                "test": f"{l_ts.item():.4f}",
                "train": f"{l_tr.item():.4f}"})
            run.log(iteration=i, test_loss=l_ts.item())  

print(f"Model Size = {sum(p.numel() for p in m.parameters()):,}")
o = optim.AdamW(m.parameters(), lr = 1e-3)

#print(f"Initial Loss: {estimate_loss(m)}")
#print("[sample]", decode(m.generate(100)))

tracker = Tracker(
    model_id="gpt-v6-test", 
    description="Testing", 
    param_count=sum(p.numel() for p in m.parameters()),
    context_width=m.context_width)

with tracker.start_run() as run:
    train(m,o,run, iterations=4000, batch_size=256)

print(f"Final Loss: {estimate_loss(m)}")
print("[sample]", decode(m.generate(100)))

# test=1.5539


Model Size = 1,591,680


Learning: 100%|██████████| 4000/4000 [01:58<00:00, 33.89it/s, test=1.5145, train=1.1270]


Final Loss: (tensor(1.4892, device='cuda:0'), tensor(1.1241, device='cuda:0'))
[sample] The ruther do smooth Capulet. More than mine honour
I'll cry 'Angelo or satisfied;' and what wicked



In [5]:
print(f"Loss: {estimate_loss(m)}")
print("[sample]", decode(m.generate(1000)))

Loss: (tensor(1.5048, device='cuda:0'), tensor(1.1032, device='cuda:0'))
[sample] 
CORIOLANUS:
A-best of breath?

HERMIONE:
And fellow. Good night:
Well, we dies well I end; not I love myself:
Our were beseech you on your tribunes as you.

QUEEN ELIZABETH:
I drown my head, 'tis short so light,
Our right begue their hate till he came.
And here be seen, Norfolk, but too long.
I pray you, she's not I no reasons can must you?
yet I cannot do't, Bushy?

Sirrah:
What wilt thou go about
The summon life dead! O, the duke's deliverance!
That I am so rise unto thee; above warriors
That shows like one that's more or slaughter,
Well apaced to bear; 'tis a ned of divine.

LADY CAPULET:
The bloody of this men made not a woman special:
sir, in him a monstrous supplicators for Saint Friar Henry,
So deave as secret folly: three humours
With a subject in the king's sake, the more,
And unseen-with our dispenses must I to Polixenes:
And so and Hereford, Warwick, time and Clarence,
That would maid the king

In [42]:
H = 8
T = 4
ang = torch.tensor([1000.0**(-2*i/H) for i in range(H//2)])

ang = torch.arange(0, T).view(-1,1) * ang
cos = torch.cos(ang)
sin = torch.sin(ang)

x = torch.ones((T,H))

xx, yy = x.chunk(2,-1)
r = torch.concat(
   [xx*cos - yy*sin,
    xx*sin + yy*cos],
    dim=-1)


print(ang)
print(x)
print(r)



tensor([[0.0000, 0.0000, 0.0000, 0.0000],
        [1.0000, 0.1778, 0.0316, 0.0056],
        [2.0000, 0.3557, 0.0632, 0.0112],
        [3.0000, 0.5335, 0.0949, 0.0169]])
tensor([[1., 1., 1., 1., 1., 1., 1., 1.],
        [1., 1., 1., 1., 1., 1., 1., 1.],
        [1., 1., 1., 1., 1., 1., 1., 1.],
        [1., 1., 1., 1., 1., 1., 1., 1.]])
tensor([[ 1.0000,  1.0000,  1.0000,  1.0000,  1.0000,  1.0000,  1.0000,  1.0000],
        [-0.3012,  0.8073,  0.9679,  0.9944,  1.3818,  1.1611,  1.0311,  1.0056],
        [-1.3254,  0.5892,  0.9348,  0.9887,  0.4932,  1.2856,  1.0612,  1.0112],
        [-1.1311,  0.3525,  0.9008,  0.9830, -0.8489,  1.3696,  1.0902,  1.0167]])
